In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
from torch_geometric.nn import GCNConv
from transformers import AutoTokenizer
from torch_geometric.utils import add_self_loops
import json
import os

# ==================================================================================
# CONFIGURATION
# ==================================================================================

# Paths (Ensure these match your setup)
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
MODEL_WEIGHTS_PATH = "eeg-text-phases6-improved.pt"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_blip.json"

# Dimensions & Settings
NUM_COLORS = 9       
NUM_OBJECTS = 6      
COLOR_NAMES = ["Black", "Blue", "Brown", "Green", "Grey", "Orange", "Red", "White", "Yellow"]
OBJECT_NAMES = ["Animal", "Building", "Food", "Nature", "Person", "Vehicle"]
ENC_HIDDEN = 256
DEC_HIDDEN = 256
DEC_LAYERS = 2
EMB_DIM = 256

# Force CPU
device = torch.device("cpu")
print(f"Running on: {device}")

# Load Tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
except:
    print("Local tokenizer not found, downloading bert-base-uncased...")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# ==================================================================================
# MODEL CLASSES (Must match training script exactly)
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]
        
        # CPU-safe edge expansion
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden

class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        self.color_processor = nn.Sequential(
            nn.Linear(num_colors, 64), nn.ReLU(), nn.Linear(64, color_feature_dim)
        )
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, object_feature_dim)
        )
        self.output_dim = color_feature_dim + object_feature_dim

    def forward(self, metadata):
        color_input = metadata[:, :NUM_COLORS].float()
        object_input = metadata[:, NUM_COLORS:].float()
        return torch.cat([self.color_processor(color_input), self.object_processor(object_input)], dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)
        self.rnn_input_dim = emb_dim + enc_dim + meta_features_dim + enc_dim
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        return bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        rnn_input = torch.cat((
            embedded,
            context.permute(1, 0, 2),
            meta_features.unsqueeze(0),
            global_eeg_context.unsqueeze(0)
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_feature_dim=32, object_feature_dim=32, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        self.meta_encoder = MetadataEncoder(num_colors, num_objects, color_feature_dim, object_feature_dim)
        
        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                               self.meta_encoder.output_dim, dec_layers, pad_id, dropout)

        enc_dim = enc_hidden * 2
        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256), nn.ReLU(), nn.LayerNorm(256), nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors

# ==================================================================================
# HELPERS
# ==================================================================================

def create_static_graph(num_channels=62):
    """Creates a fully connected graph for inference stability on CPU."""
    edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
    edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
    edge_attr = torch.ones(edge_index.shape[1], dtype=torch.float)
    return edge_index, edge_attr

def beam_search_decoder(model, eeg_signal, meta_signal, edge_index, edge_attr, beam_width=3, max_len=30):
    model.eval()
    with torch.no_grad():
        eeg_signal = eeg_signal.unsqueeze(0).to(device)
        meta_signal = meta_signal.unsqueeze(0).to(device)
        
        encoder_outputs, encoder_hidden = model.encoder(eeg_signal, edge_index, edge_attr)
        meta_features = model.meta_encoder(meta_signal)
        decoder_hidden = model.decoder.init_hidden(encoder_hidden)
        
        hidden_reshaped = encoder_hidden.view(model.encoder.rnn.num_layers, 2, 1, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        
        meta_preds = model.meta_head(global_eeg_context)
        pred_color_ids = (torch.sigmoid(meta_preds[0, :model.num_colors]) > 0.5).nonzero(as_tuple=True)[0].tolist()
        pred_object_ids = (torch.sigmoid(meta_preds[0, model.num_colors:]) > 0.5).nonzero(as_tuple=True)[0].tolist()

        beams = [(0.0, SOS_ID, decoder_hidden, [])]
        
        for _ in range(max_len):
            candidates = []
            for score, input_id, hidden, seq in beams:
                if len(seq) > 0 and seq[-1] == EOS_ID:
                    candidates.append((score, input_id, hidden, seq))
                    continue
                
                token_tensor = torch.tensor([input_id], device=device)
                prediction, new_hidden, _ = model.decoder(
                    token_tensor, hidden, encoder_outputs, meta_features, global_eeg_context
                )
                
                log_probs = F.log_softmax(prediction, dim=-1).squeeze(0)
                topk_probs, topk_ids = log_probs.topk(beam_width)
                
                for k in range(beam_width):
                    next_score = score + topk_probs[k].item()
                    next_id = topk_ids[k].item()
                    candidates.append((next_score, next_id, new_hidden, seq + [next_id]))
            
            beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(seq[-1] == EOS_ID for _, _, _, seq in beams if len(seq) > 0):
                break

        best_score, _, _, best_seq = beams[0]
        if best_seq and best_seq[-1] == EOS_ID: best_seq = best_seq[:-1]
        
        return tokenizer.decode(best_seq, skip_special_tokens=True), pred_color_ids, pred_object_ids

# ==================================================================================
# MAIN EXECUTION
# ==================================================================================
if __name__ == "__main__":
    if not os.path.exists(MODEL_WEIGHTS_PATH):
        print(f"Error: Weights file '{MODEL_WEIGHTS_PATH}' not found.")
        exit()

    # 1. Load Model
    print("Loading model on CPU...")
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        enc_hidden=ENC_HIDDEN,
        dec_hidden=DEC_HIDDEN,
        emb_dim=EMB_DIM,
        dec_layers=DEC_LAYERS
    ).to(device)

    # Load weights with CPU mapping
    state_dict = torch.load(MODEL_WEIGHTS_PATH, map_location=device)
    model.load_state_dict(state_dict)
    print("Model loaded.")

    # 2. Create Graph (Static Fallback for CPU)
    edge_index, edge_attr = create_static_graph(num_channels=62)
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)

    # 3. Load Data & Run Inference
    print("Loading dataset sample...")
    try:
        with h5py.File(H5_FILE_PATH, 'r') as f:
            total_samples = f['eeg'].shape[0]
    
            print(f"\nRunning inference on 20 random samples...\n")
    
            for n in range(20):
                idx = np.random.randint(0, total_samples)
    
                eeg = torch.from_numpy(f['eeg'][idx].astype(np.float32))
                meta = torch.from_numpy(f['metadata'][idx].astype(np.float32))
                true_text_ids = f['input_ids'][idx].astype(np.int64)
    
                true_text = tokenizer.decode(true_text_ids, skip_special_tokens=True)

                # ---------------------------
                # Decode Ground-Truth Metadata
                # ---------------------------
                meta_np = meta.numpy()
                
                gt_color_ids = np.where(meta_np[:NUM_COLORS] == 1)[0].tolist()
                gt_object_ids = np.where(meta_np[NUM_COLORS:] == 1)[0].tolist()
                
                gt_colors = [COLOR_NAMES[i] for i in gt_color_ids]
                gt_objects = [OBJECT_NAMES[i] for i in gt_object_ids]
                
                print(f"\n================ SAMPLE {n+1} / 20 ================")
                print(f"Sample Index: {idx}")
                print(f"Ground Truth Text: {true_text}")
                print(f"GT Colors:  {gt_colors}")
                print(f"GT Objects: {gt_objects}")

                # Run Inference
                pred_text, pred_colors, pred_objects = beam_search_decoder(
                    model, eeg, meta, edge_index, edge_attr, beam_width=5
                )
    
                # Decode metadata
                p_colors = [COLOR_NAMES[i] for i in pred_colors if i < len(COLOR_NAMES)]
                p_objects = [OBJECT_NAMES[i] for i in pred_objects if i < len(OBJECT_NAMES)]
    
                print(f"Prediction:   {pred_text}")
                print(f"Pred Colors:  {p_colors}")
                print(f"Pred Objects: {p_objects}")
    
    except Exception as e:
        print(f"Error during inference: {e}")

Running on: cpu
Loading model on CPU...
Model loaded.
Loading dataset sample...

Running inference on 20 random samples...


================ SAMPLE 1 / 20 ================
Sample Index: 4346
Ground Truth Text: a red watermelon is seen in this image
GT Colors:  ['Green', 'Red']
GT Objects: ['Food', 'Nature']
Prediction:   a bunch of bananas
Pred Colors:  ['Blue', 'Orange', 'Red', 'Yellow']
Pred Objects: ['Animal', 'Building', 'Food', 'Nature']

================ SAMPLE 2 / 20 ================
Sample Index: 13587
Ground Truth Text: a panda eating bamboo leaves in a zoo
GT Colors:  ['Black', 'Green', 'White']
GT Objects: ['Animal', 'Nature']
Prediction:   a turtle swimming in the ocean
Pred Colors:  ['Black', 'Blue', 'Green', 'Grey', 'White']
Pred Objects: ['Animal', 'Building', 'Nature', 'Person', 'Vehicle']

================ SAMPLE 3 / 20 ================
Sample Index: 3957
Ground Truth Text: many horses walking on a beach
GT Colors:  ['Brown', 'Green', 'Grey']
GT Objects: ['Animal', 'N